# 08 - Phase 2: structural vs content (the mechanism)
The panel shows detectors miss hijack injections; Phase 1 shows those misses are real exploits. **Why** do they miss them? Hypothesis: the detectors key on harmful *surface content*, not on the *structural fact* that an instruction was injected.

Controlled 2x2: factor A = payload (harmful-worded vs benign-hijack), factor B = structure (standalone prompt vs embedded in a host document). Same instructions, same host docs, only the two factors vary.

- A content-keyed detector: p driven by payload (A), ignores structure (B).
- A structure-keyed (ideal injection) detector: p driven by structure (B) -- fires on embedded instructions regardless of payload.

**Decisive cell:** benign-hijack + embedded is a *real injection with a benign payload*. An ideal detector flags it; a content detector misses it. If detectors miss it while flagging harmful-standalone (not an injection), the mechanism is content-keying, proven.

## Bootstrap + deps

In [1]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
print('Session ready.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready.


In [2]:
!pip install -q transformers sentencepiece
import torch; print('CUDA:', torch.cuda.is_available())

CUDA: True


In [4]:
import importlib, data_loaders, detectors, subprocess
for m in (data_loaders, detectors): importlib.reload(m)
from data_loaders import _bipia_attacks_by_cat, _bipia_contexts, BIPIA_HARMFUL, BIPIA_HIJACK
from detectors import score_released, threshold_at_fpr
import numpy as np, pandas as pd, os, random

WORK='/content/_bipia'
if not os.path.exists(WORK):
    print('cloning BIPIA...')
    subprocess.run(['git','clone','--depth','1','https://github.com/microsoft/BIPIA.git',WORK],check=False)

cats = _bipia_attacks_by_cat(WORK)
ctxs = _bipia_contexts(WORK); random.Random(0).shuffle(ctxs)
HOSTS = ctxs[:5]                       # 5 fixed host docs (structure control)
def uniq(catset):
    out=[]
    for c in catset:
        out += [a for a in cats.get(c,[]) if 3<len(a)<400]
    return list(dict.fromkeys(out))[:20]   # up to 20 unique instructions per payload
harm = uniq(BIPIA_HARMFUL); hij = uniq(BIPIA_HIJACK)
print('harmful instrs', len(harm), '| hijack instrs', len(hij), '| hosts', len(HOSTS))

rows=[]
for payload,instrs in [('harmful',harm),('benign_hijack',hij)]:
    for ins in instrs:
        rows.append({'text':ins,'payload':payload,'structure':'standalone'})
        for h in HOSTS:
            rows.append({'text':h+'\n'+ins,'payload':payload,'structure':'embedded'})
for h in HOSTS:
    rows.append({'text':h,'payload':'none','structure':'clean_doc'})
cond = pd.DataFrame(rows)
print(cond.groupby(['payload','structure']).size())

cloning BIPIA...
harmful instrs 10 | hijack instrs 20 | hosts 5
payload        structure 
benign_hijack  embedded      100
               standalone     20
harmful        embedded       50
               standalone     10
none           clean_doc       5
dtype: int64


## Build the 2x2 conditions (same instructions, same host docs)

In [5]:
import importlib, data_loaders, detectors
for m in (data_loaders, detectors): importlib.reload(m)
from data_loaders import _bipia_attacks_by_cat, _bipia_contexts, BIPIA_HARMFUL, BIPIA_HIJACK
from detectors import score_released, threshold_at_fpr
import numpy as np, pandas as pd, os, random

WORK='/content/_bipia'
cats = _bipia_attacks_by_cat(WORK)
ctxs = _bipia_contexts(WORK); random.Random(0).shuffle(ctxs)
HOSTS = ctxs[:5]                       # 5 fixed host docs (structure control)
def uniq(catset):
    out=[]
    for c in catset:
        out += [a for a in cats.get(c,[]) if 3<len(a)<400]
    return list(dict.fromkeys(out))[:20]   # up to 20 unique instructions per payload
harm = uniq(BIPIA_HARMFUL); hij = uniq(BIPIA_HIJACK)
print('harmful instrs', len(harm), '| hijack instrs', len(hij), '| hosts', len(HOSTS))

rows=[]
for payload,instrs in [('harmful',harm),('benign_hijack',hij)]:
    for ins in instrs:
        rows.append({'text':ins,'payload':payload,'structure':'standalone'})
        for h in HOSTS:
            rows.append({'text':h+'\n'+ins,'payload':payload,'structure':'embedded'})
# clean host-doc reference (no injection)
for h in HOSTS:
    rows.append({'text':h,'payload':'none','structure':'clean_doc'})
cond = pd.DataFrame(rows)
print(cond.groupby(['payload','structure']).size())

harmful instrs 10 | hijack instrs 20 | hosts 5
payload        structure 
benign_hijack  embedded      100
               standalone     20
harmful        embedded       50
               standalone     10
none           clean_doc       5
dtype: int64


## Score the 3 detectors on all conditions

In [ ]:
DETECTORS={'protectai_v2':'protectai/deberta-v3-base-prompt-injection-v2',
           'prompt_guard_2':'meta-llama/Llama-Prompt-Guard-2-86M',
           'prompt_guard_2_22m':'meta-llama/Llama-Prompt-Guard-2-22M'}
from huggingface_hub import login; import getpass
_t=getpass.getpass('HF token (for Prompt-Guard; blank=ProtectAI only): ').strip()
if _t: login(token=_t)
os.makedirs('data/phase2',exist_ok=True)
P={}
for tag,name in DETECTORS.items():
    fp=f'data/phase2/score_{tag}.npy'
    try:
        P[tag]=np.load(fp) if os.path.exists(fp) else score_released(cond.text.tolist(),name)
        if not os.path.exists(fp): np.save(fp,P[tag])
        print(tag,'ok')
    except Exception as e: print('[skip]',tag,e)

HF token (for Prompt-Guard; blank=ProtectAI only): ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/994 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  protectai/deberta-v3-base-prompt-injection-v2: id2label={0: 'SAFE', 1: 'INJECTION'} -> attack class index 1


## Cell means + main effects
Mean p per (detector x payload x structure). The two main effects: how much p moves with payload (content) vs with structure.

In [ ]:
def boot_ci(x,B=1000,seed=0):
    r=np.random.default_rng(seed); x=np.asarray(x)
    if len(x)==0: return (np.nan,np.nan)
    m=[x[r.integers(0,len(x),len(x))].mean() for _ in range(B)]
    return tuple(np.percentile(m,[2.5,97.5]))

cells=[]
for tag,p in P.items():
    cond['_p']=p
    for (pay,st),g in cond.groupby(['payload','structure']):
        lo,hi=boot_ci(g['_p'].values)
        cells.append({'detector':tag,'payload':pay,'structure':st,'n':len(g),
                      'mean_p':round(float(g['_p'].mean()),3),'ci_lo':round(lo,3),'ci_hi':round(hi,3)})
cells=pd.DataFrame(cells)
print(cells.to_string(index=False))

In [ ]:
# main effects per detector: content effect (harmful-embedded - benign_hijack-embedded),
# structure effect (benign_hijack: embedded - standalone)
eff=[]
for tag,p in P.items():
    cond['_p']=p
    def mp(pay,st):
        m=cond[(cond.payload==pay)&(cond.structure==st)]['_p']; return float(m.mean()) if len(m) else np.nan
    content = mp('harmful','embedded') - mp('benign_hijack','embedded')
    structure_hij = mp('benign_hijack','embedded') - mp('benign_hijack','standalone')
    structure_harm = mp('harmful','embedded') - mp('harmful','standalone')
    eff.append({'detector':tag,
        'content_effect(harm-hijack|embedded)':round(content,3),
        'structure_effect_hijack(emb-alone)':round(structure_hij,3),
        'structure_effect_harmful(emb-alone)':round(structure_harm,3),
        'p_benignhijack_embedded':round(mp('benign_hijack','embedded'),3),
        'p_harmful_standalone':round(mp('harmful','standalone'),3)})
effdf=pd.DataFrame(eff); print(effdf.to_string(index=False))

## Figure: 2x2 mean p per detector

In [ ]:
import matplotlib.pyplot as plt
dets=list(P.keys()); fig,axes=plt.subplots(1,len(dets),figsize=(5*len(dets),4),squeeze=False)
for ax,tag in zip(axes[0],dets):
    cond['_p']=P[tag]
    sts=['standalone','embedded']; pays=['benign_hijack','harmful']
    x=np.arange(len(sts)); w=0.35
    for k,pay in enumerate(pays):
        vals=[cond[(cond.payload==pay)&(cond.structure==s)]['_p'].mean() for s in sts]
        ax.bar(x+k*w,vals,w,label=pay)
    ax.set_xticks(x+w/2); ax.set_xticklabels(sts); ax.set_ylim(0,1)
    ax.set_title(tag); ax.set_ylabel('mean p(attack)'); ax.legend()
plt.tight_layout(); os.makedirs('figures',exist_ok=True)
plt.savefig('figures/phase2_structure_vs_content.png',dpi=150); plt.show()

## Verdict + persist + commit

In [ ]:
from reslog import log_result
v=['STRUCTURE vs CONTENT:']
for _,r in effdf.iterrows():
    ce=abs(r['content_effect(harm-hijack|embedded)']); se=abs(r['structure_effect_hijack(emb-alone)'])
    tag=r['detector']
    v.append(f"[{tag}] content_effect={r['content_effect(harm-hijack|embedded)']}, structure_effect(hijack)={r['structure_effect_hijack(emb-alone)']}")
    if ce>se+0.05: v.append('   -> p driven by PAYLOAD not structure: CONTENT-KEYED (misses benign-payload injections).')
    elif se>ce+0.05: v.append('   -> p driven by STRUCTURE: some injection-structure awareness.')
    else: v.append('   -> weak on both; near-flat response.')
    v.append(f"   benign-hijack EMBEDDED (a real injection) scored p={r['p_benignhijack_embedded']} (low = missed).")
verdict='\n'.join(v); print(verdict)
cells.to_csv('reports/phase2_cell_means.csv',index=False); effdf.to_csv('reports/phase2_effects.csv',index=False)
log_result('Phase 2 structure-vs-content mechanism', cells.to_string(index=False)+'\n\n'+effdf.to_string(index=False)+'\n\n'+verdict, csv_df=effdf, csv_name='phase2_effects.csv')
!git add -A && git commit -m "Phase 2: structure-vs-content controlled experiment (mechanism)" && git push
print('done')